# SmolVLA Memory SFT — Kaggle 2×T4

- **Accelerator**: Chọn **GPU T4 ×2** (Dual GPU)
- **Internet**: Chuyển sang **On** (để cài packages và tải checkpoint/tokenizer)
- **Input**: Attach dataset chứa `memory_sft_code.zip` và dataset dữ liệu đã chuẩn bị (nếu có).


## 1. Cài đặt thư viện trực tiếp vào môi trường Kaggle
Kaggle đã cài sẵn PyTorch GPU + CUDA cho card 2×T4. Cài đặt trực tiếp các thư viện bổ trợ của LeRobot và SFT (không dùng venv để tránh lỗi ensurepip/xung đột NumPy):


In [ ]:
!pip install --upgrade pip
!pip install "lerobot[smolvla,peft]>=0.4.0" "peft>=0.12.0" "accelerate>=0.34.0" "transformers>=4.48.0" "h5py>=3.10.0" "huggingface-hub>=0.25.0"


## 2. Tải Checkpoint gốc và Tokenizer (Assets)
Tải checkpoint `lerobot/smolvla_libero` và tokenizer `SmolVLM2-500M-Video-Instruct` đã được pin commit và xác thực mã băm SHA256:


In [ ]:
from pathlib import Path
ASSETS_DIR = Path("/kaggle/temp/sft-assets")

!python scripts/memory_sft_assets.py fetch {ASSETS_DIR}


## 3. Chọn và thiết lập Dữ liệu (Dataset)
- Khuyến nghị: Upload dataset đã chuẩn bị (có `manifest.json` và `shards/`) vào `/kaggle/input`.
- Hoặc chuẩn bị trực tiếp trên Kaggle: đặt `PREPARE_ON_KAGGLE = True`.


In [ ]:
import json
from pathlib import Path

PREPARE_ON_KAGGLE = False
DATA_DIR = Path("/kaggle/input/libero10-memory")
TASK_NAMES = []  # ví dụ: ["STUDY_SCENE1_pick_up_the_book_and_place_it_in_the_back_compartment_of_the_caddy"]

if PREPARE_ON_KAGGLE:
    DATA_DIR = Path("/kaggle/working/libero10-memory")
    extra = " ".join([f"--task-names {t}" for t in TASK_NAMES])
    !python scripts/prepare_memory_sft.py --suite libero_10 --source-root /kaggle/temp/libero-raw --output-dir {DATA_DIR} --tokenizer-path {ASSETS_DIR}/backbone --download --source-lock resources/manifests/libero_10_sft_sources.json {extra}
else:
    # Tự động tìm manifest.json trong /kaggle/input nếu DATA_DIR mặc định chưa khớp
    if not (DATA_DIR / "manifest.json").is_file():
        manifests = list(Path("/kaggle/input").rglob("manifest.json"))
        if manifests:
            DATA_DIR = manifests[0].parent
            print(f"Auto-detected dataset tại: {DATA_DIR}")

if not (DATA_DIR / "manifest.json").is_file():
    raise FileNotFoundError("Không tìm thấy manifest.json! Hãy attach dataset đã chuẩn bị hoặc đặt PREPARE_ON_KAGGLE = True.")

manifest = json.loads((DATA_DIR / "manifest.json").read_text())
print("Suite:", manifest.get("suite", "unknown"), "| Commit:", manifest.get("source_revision", "unknown"))
for row in manifest.get("sources", []):
    print("Task:", row.get("path"), "| Split frames:", row.get("frame_memory_counts_by_split"))


## 4. Smoke test trên 2× GPU T4 (2 optimizer updates)
Xác nhận huấn luyện song song Accelerate DDP phân bổ đúng 2 GPU, FP16 AMP không bị OOM trước khi chạy dài:


In [ ]:
SMOKE_OUTPUT = "/kaggle/working/sft-smoke"

!accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision fp16 --dynamo_backend no \
  scripts/train_memory_sft.py \
  --assets-dir {ASSETS_DIR} \
  --data-dir {DATA_DIR} \
  --output-dir {SMOKE_OUTPUT} \
  --expected-gpus 2 \
  --steps 2 \
  --warmup-steps 0 \
  --validation-max-batches 2 \
  --save-every 1


## 5. Huấn luyện chính thức (Training)
Preset: frozen VLM backbone, expert LoRA rank 16 / alpha 16, FP16, batch 2/GPU x accumulation 8 = effective batch 32.


In [ ]:
RUN_TRAINING = False  # Đổi thành True khi smoke test ở bước 4 đã qua
CONDITION = "memory"   # 'memory' hoặc 'no_memory' (đối chứng)
OUTPUT_DIR = f"/kaggle/working/sft-{CONDITION}"

if RUN_TRAINING:
    !accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision fp16 --dynamo_backend no \
      scripts/train_memory_sft.py \
      --assets-dir {ASSETS_DIR} \
      --data-dir {DATA_DIR} \
      --output-dir {OUTPUT_DIR} \
      --expected-gpus 2 \
      --condition {CONDITION} \
      --adaptation expert_lora \
      --steps 3000


## 6. Xuất và tải Checkpoint về máy Local
Mô hình đã merge LoRA sẽ nằm tại: `{OUTPUT_DIR}/pretrained_model/`.
Tải thư mục này về máy local và kiểm định bằng:
```bash
python scripts/validate_memory_sft_export.py --checkpoint /path/to/pretrained_model --data-dir experiments/pilot_sft_test
```
